In [19]:
import openai
from qdrant_client import QdrantClient

### Embedding function

In [20]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval function

In [21]:
qdrant_client = QdrantClient(url="http://localhost:6333")

/var/folders/r_/3061nb0s7px95mfb__nvwjsc0000gn/T/ipykernel_54432/3504550968.py:1: UserWarning: Qdrant client version 1.16.1 is incompatible with server version 1.18.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(url="http://localhost:6333")


In [22]:
def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name = "Amazon-items-collection-00",
        query = query_embedding,
        limit = k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [23]:
retrieved_context = retrieve_data("What kind of earphones can I get?", qdrant_client, k=10)

In [24]:
retrieved_context

{'retrieved_context_ids': ['B08VQYCNHB',
  'B00NF15R2W',
  'B09M8CC81H',
  'B089Q4334P',
  'B08ZN4GW2F',
  'B00FZ1PCK2',
  'B00PUUWJ8G',
  'B00J3J3KNI',
  'B003XX6ZIM',
  'B00G2JF3E6'],
 'retrieved_context': ['Unplugged (CD) ',
  'BEE GEES ODESSA MINI LP CD OBI ',
  'IVE ELEVEN 1st Single Album Random Version CD+1p Folding Poster On Pack+92p PhotoBook+1p PhotoCard+Tracking Sealed ',
  'STRAY KIDS GO生 1st Album LIMITED CD+Photo Book+5 Card+Film+Lyric+PreOrder+TRACKING CODE K-POP SEALED ',
  'ASTRO [ ALL YOURS ] 2nd Album 3 Ver FULL SET. 3 CD+3 Photo Book(each 104p)+3 Accordion Post Card(each 6p) 3 Message Card+3 Photo Card+3 Folded Poster(On pack) K-POP SEALED ',
  'The Songs of Distant Earth ',
  'Pink Friday [Deluxe Edition] by Nicki Minaj [Music CD] ',
  'Enrique Iglesias- Sex and Love Deluxe Extended Edition with 4 Bonus Tracks ',
  'GORDON LIGHTFOOT - endless wire WB 3149 (LP vinyl record) ',
  'Best Of Dire Straits & Mark Knopfler: Private Investigations (2CD) by Dire Straits, Mar

### Format retreived context function

In [32]:
def process_context(context):
    formatted_context = ""
    for chunk_id, chunk, score in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["similarity_scores"]
    ):
        formatted_context += f"- ID: {chunk_id}, similarity: {score:.4f}, description: {chunk}\n"
    return formatted_context

In [33]:
preprocessed_context = process_context(retrieved_context)

In [34]:
print(preprocessed_context)

- ID: B08VQYCNHB, similarity: 0.2073, description: Unplugged (CD) 
- ID: B00NF15R2W, similarity: 0.1890, description: BEE GEES ODESSA MINI LP CD OBI 
- ID: B09M8CC81H, similarity: 0.1807, description: IVE ELEVEN 1st Single Album Random Version CD+1p Folding Poster On Pack+92p PhotoBook+1p PhotoCard+Tracking Sealed 
- ID: B089Q4334P, similarity: 0.1702, description: STRAY KIDS GO生 1st Album LIMITED CD+Photo Book+5 Card+Film+Lyric+PreOrder+TRACKING CODE K-POP SEALED 
- ID: B08ZN4GW2F, similarity: 0.1623, description: ASTRO [ ALL YOURS ] 2nd Album 3 Ver FULL SET. 3 CD+3 Photo Book(each 104p)+3 Accordion Post Card(each 6p) 3 Message Card+3 Photo Card+3 Folded Poster(On pack) K-POP SEALED 
- ID: B00FZ1PCK2, similarity: 0.1595, description: The Songs of Distant Earth 
- ID: B00PUUWJ8G, similarity: 0.1570, description: Pink Friday [Deluxe Edition] by Nicki Minaj [Music CD] 
- ID: B00J3J3KNI, similarity: 0.1514, description: Enrique Iglesias- Sex and Love Deluxe Extended Edition with 4 Bonus T

### Create Prompt function

In [35]:
def build_prompt(preprocessed_context, question):
    prompt = f"""
        You are a shopping assistant that can answer questions about the products in stock.

        You will be given a question and list of context

        Instructions:
        - You need to answer question based on the provided context only.
        - Never use word context and refer to it as the available products.

        Context:
        {preprocessed_context}

        Question:
        {question}
    """

    return prompt


In [37]:
prompt = build_prompt(preprocessed_context, "What kind of earphone can I get?")

In [38]:
print(prompt)


        You are a shopping assistant that can answer questions about the products in stock.

        You will be given a question and list of context

        Instructions:
        - You need to answer question based on the provided context only.
        - Never use word context and refer to it as the available products.

        Context:
        - ID: B08VQYCNHB, similarity: 0.2073, description: Unplugged (CD) 
- ID: B00NF15R2W, similarity: 0.1890, description: BEE GEES ODESSA MINI LP CD OBI 
- ID: B09M8CC81H, similarity: 0.1807, description: IVE ELEVEN 1st Single Album Random Version CD+1p Folding Poster On Pack+92p PhotoBook+1p PhotoCard+Tracking Sealed 
- ID: B089Q4334P, similarity: 0.1702, description: STRAY KIDS GO生 1st Album LIMITED CD+Photo Book+5 Card+Film+Lyric+PreOrder+TRACKING CODE K-POP SEALED 
- ID: B08ZN4GW2F, similarity: 0.1623, description: ASTRO [ ALL YOURS ] 2nd Album 3 Ver FULL SET. 3 CD+3 Photo Book(each 104p)+3 Accordion Post Card(each 6p) 3 Message Card+3 Photo 

#### Generate answer function

In [39]:
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="minimal"
    )

    return response.choices[0].message.content


In [40]:
print(generate_answer(prompt))

From the available products, there aren’t any earphones listed. If you’re looking for earphones, I can help check if we have any in stock or suggest similar audio-related items (like CDs or headphones) from the current inventory. Would you like me to search for earphones specifically or suggest related items?


#### Combined RAG Pipeline

In [41]:
def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")
    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)


    return answer

In [42]:
print(rag_pipeline("What kind of music can I listen to with above 4.5 score?"))

/var/folders/r_/3061nb0s7px95mfb__nvwjsc0000gn/T/ipykernel_54432/4206078479.py:2: UserWarning: Qdrant client version 1.16.1 is incompatible with server version 1.18.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(url="http://localhost:6333")


From the available products, none list a 4.5 score specifically. The items shown have various descriptions, but no item is labeled with a 4.5 score in the provided list. If you’re looking for music recommendations, I can summarize what each listed item is:

- 5 Classic Albums
- The Songs of Distant Earth
- Enrique Iglesias - Sex and Love Deluxe Extended Edition
- Complete Classic Rock Collection 8 CD Box
- Bon Jovi - Gonna Set The World On Fire: The Legendary Broadcasts 1983-1993

If you can share which score you’re referring to, I can help identify the closest matches from the available products.
